# Notebook 19: Variance Estimation for Pathway Nulls

## Purpose
Compute variance of pathway counts from held-out permutations for z-score calculation.

**Goal**: Enable z = (observed - expected) / √variance for anomaly detection

## Method
1. Use permutations 21-30 for variance estimation (independent from training 1-20)
2. For each metapath, compute actual pathway counts in each permutation
3. Compute variance across permutations for each (source, target) pair
4. For sparse pairs with insufficient samples, use adaptive degree binning
5. Save variance matrices and lookup tables

## Inputs
- data/hetionet-v1.0/hetmat/edges/*.sparse.npz (permutations 21-30)
- results/compositional_validation/validation_summary.json

## Outputs
- results/variance_estimates/{metapath}_variance.npz (sparse variance matrix)
- results/variance_estimates/{metapath}_variance_lookup.pkl (degree bin pooling)
- results/variance_estimates/summary.csv

## Dependencies
- Notebook 17 must PASS (compositional calculation validated)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
from scipy.stats import describe
import json
import pickle
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

repo_dir = Path.cwd().parent
data_dir = repo_dir / 'data'
results_dir = repo_dir / 'results' / 'variance_estimates'
results_dir.mkdir(parents=True, exist_ok=True)

print(f"Repository: {repo_dir}")
print(f"Results will be saved to: {results_dir}")

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## Check Prerequisites

In [ ]:
# Check compositional validation passed
validation_file = repo_dir / 'results' / 'compositional_validation' / 'validation_summary.json'

if not validation_file.exists():
    raise FileNotFoundError(
        f"Validation file not found: {validation_file}\n"
        "Please run notebook 17_compositional_validation.ipynb first!"
    )

with open(validation_file, 'r') as f:
    validation_summary = json.load(f)

if validation_summary['overall_decision'] == 'FAILED':
    raise ValueError(
        "Compositional validation FAILED!\n"
        "Variance estimation not meaningful without compositional approach."
    )

print("✓ Prerequisites check passed")
print(f"  Compositional validation: {validation_summary['overall_decision']}")
print(f"  Mean Pearson r: {validation_summary['overall_mean_pearson_r']:.4f}")

## Configuration

In [ ]:
# Papermill parameters
variance_perms_start = 21
variance_perms_end = 30
min_samples_per_bin = 30
random_seed = 42

In [ ]:
# Metapaths to compute variance for
metapaths_2hop = [
    {'name': 'CbGpPW', 'edge1': 'CbG', 'edge2': 'GpPW'},
    {'name': 'CtDaG', 'edge1': 'CtD', 'edge2': 'DaG'},
    {'name': 'CbGaD', 'edge1': 'CbG', 'edge2': 'GaD'},
    {'name': 'CrCbG', 'edge1': 'CrC', 'edge2': 'CbG'},
    {'name': 'CbGiG', 'edge1': 'CbG', 'edge2': 'GiG'},
    {'name': 'CpDaG', 'edge1': 'CpD', 'edge2': 'DaG'},
    {'name': 'CbGpBP', 'edge1': 'CbG', 'edge2': 'GpBP'},
    {'name': 'CbGpCC', 'edge1': 'CbG', 'edge2': 'GpCC'},
]

print(f"Computing variance for {len(metapaths_2hop)} metapaths")
print(f"Variance permutations: {variance_perms_start}-{variance_perms_end}")
print(f"Minimum samples per bin: {min_samples_per_bin}")

## Helper Functions

In [ ]:
def load_edge_matrix(edge_type, perm_id, base_dir='hetionet-v1.0'):
    """
    Load edge matrix for a specific permutation.
    
    Args:
        edge_type: Edge type abbreviation
        perm_id: Permutation ID (1-200)
        base_dir: Base directory name
    
    Returns:
        scipy.sparse matrix
    """
    perm_dir = f'{perm_id:03d}.hetmat'
    edge_file = data_dir / base_dir / 'permutations' / perm_dir / 'edges' / f'{edge_type}.sparse.npz'
    
    if not edge_file.exists():
        raise FileNotFoundError(f"Edge file not found: {edge_file}")
    
    return sp.load_npz(str(edge_file))

def compute_metapath_matrix(perm_id, edge1_type, edge2_type):
    """
    Compute 2-hop metapath matrix for a permutation.
    
    Args:
        perm_id: Permutation ID
        edge1_type: First edge type
        edge2_type: Second edge type
    
    Returns:
        scipy.sparse matrix: Metapath counts
    """
    edge1 = load_edge_matrix(edge1_type, perm_id)
    edge2 = load_edge_matrix(edge2_type, perm_id)
    return edge1 @ edge2

def create_adaptive_bins(degrees, min_samples=30):
    """
    Create adaptive degree bins with minimum samples per bin.
    
    Args:
        degrees: Array of node degrees
        min_samples: Minimum samples per bin
    
    Returns:
        list: List of (bin_min, bin_max) tuples
    """
    unique_degrees = np.unique(degrees[degrees > 0])
    degree_counts = {d: np.sum(degrees == d) for d in unique_degrees}
    
    bins = []
    current_bin_start = unique_degrees[0]
    current_count = 0
    
    for deg in unique_degrees:
        current_count += degree_counts[deg]
        
        if current_count >= min_samples:
            bins.append((current_bin_start, deg))
            current_bin_start = deg + 1 if deg < unique_degrees[-1] else deg
            current_count = 0
    
    # Add remaining degrees to last bin
    if current_count > 0 and len(bins) > 0:
        bins[-1] = (bins[-1][0], unique_degrees[-1])
    elif len(bins) == 0:
        # If no bins created, make one bin with all degrees
        bins.append((unique_degrees[0], unique_degrees[-1]))
    
    return bins

def get_bin_index(degree, bins):
    """
    Find which bin a degree belongs to.
    
    Args:
        degree: Node degree
        bins: List of (bin_min, bin_max) tuples
    
    Returns:
        int: Bin index, or -1 if not found
    """
    for i, (bin_min, bin_max) in enumerate(bins):
        if bin_min <= degree <= bin_max:
            return i
    return -1

print("Helper functions loaded")

## Compute Variance for Each Metapath

In [ ]:
variance_perms = list(range(variance_perms_start, variance_perms_end + 1))
n_variance_perms = len(variance_perms)

print(f"Using {n_variance_perms} permutations for variance estimation: {min(variance_perms)}-{max(variance_perms)}\n")

all_variance_stats = []

for mp_info in metapaths_2hop:
    metapath = mp_info['name']
    edge1 = mp_info['edge1']
    edge2 = mp_info['edge2']
    
    print(f"\n{'='*70}")
    print(f"COMPUTING VARIANCE: {metapath}")
    print(f"  Edges: {edge1} → {edge2}")
    print(f"{'='*70}\n")
    
    # Step 1: Collect pathway samples across variance permutations
    print(f"[1/4] Collecting pathway samples...")
    pathway_samples = []
    
    for perm_id in variance_perms:
        pathways = compute_metapath_matrix(perm_id, edge1, edge2)
        pathway_samples.append(pathways)
        print(f"  Perm {perm_id}: shape {pathways.shape}, nnz={pathways.nnz:,}")
    
    # Step 2: Compute element-wise variance
    print(f"\n[2/4] Computing element-wise variance...")
    
    # Get shape
    shape = pathway_samples[0].shape
    
    # Convert all samples to dense (for variance calculation)
    pathway_arrays = np.stack([p.toarray() for p in pathway_samples], axis=0)
    
    # Compute variance across permutations (axis=0)
    variance_array = np.var(pathway_arrays, axis=0)
    
    # Convert back to sparse
    variance_matrix = sp.csr_matrix(variance_array)
    
    print(f"  Variance matrix shape: {variance_matrix.shape}")
    print(f"  Non-zero variances: {variance_matrix.nnz:,}")
    print(f"  Variance range: [{variance_matrix.data.min():.4f}, {variance_matrix.data.max():.4f}]")
    print(f"  Mean variance (non-zero): {variance_matrix.data.mean():.4f}")
    
    # Step 3: Create adaptive degree bins for pooled variance
    print(f"\n[3/4] Creating adaptive degree bins...")
    
    # Load Hetionet to get degrees
    hetionet_dir = data_dir / 'hetionet-v1.0' / 'hetmat'
    hetionet_edge1 = sp.load_npz(str(hetionet_dir / 'edges' / f'{edge1}.sparse.npz'))
    hetionet_edge2 = sp.load_npz(str(hetionet_dir / 'edges' / f'{edge2}.sparse.npz'))
    
    source_degrees = np.array(hetionet_edge1.sum(axis=1)).flatten()
    target_degrees = np.array(hetionet_edge2.sum(axis=0)).flatten()
    
    source_bins = create_adaptive_bins(source_degrees, min_samples=min_samples_per_bin)
    target_bins = create_adaptive_bins(target_degrees, min_samples=min_samples_per_bin)
    
    print(f"  Source bins: {len(source_bins)}")
    print(f"  Target bins: {len(target_bins)}")
    
    # Step 4: Compute pooled variance within bins
    print(f"\n[4/4] Computing pooled variance within bins...")
    
    variance_lookup = {}
    
    for src_bin_idx, (src_min, src_max) in enumerate(source_bins):
        for tgt_bin_idx, (tgt_min, tgt_max) in enumerate(target_bins):
            # Find all (i,j) pairs in this bin
            src_mask = (source_degrees >= src_min) & (source_degrees <= src_max)
            tgt_mask = (target_degrees >= tgt_min) & (target_degrees <= tgt_max)
            
            src_indices = np.where(src_mask)[0]
            tgt_indices = np.where(tgt_mask)[0]
            
            # Extract variance values for this bin
            bin_variances = []
            for i in src_indices:
                for j in tgt_indices:
                    if variance_matrix[i, j] > 0:
                        bin_variances.append(variance_matrix[i, j])
            
            if len(bin_variances) > 0:
                pooled_var = np.mean(bin_variances)
            else:
                pooled_var = 0.0
            
            variance_lookup[(src_bin_idx, tgt_bin_idx)] = {
                'variance': pooled_var,
                'n_samples': len(bin_variances),
                'src_bin': (src_min, src_max),
                'tgt_bin': (tgt_min, tgt_max)
            }
    
    print(f"  Created {len(variance_lookup)} bin combinations")
    
    # Save variance matrix
    variance_file = results_dir / f'{metapath}_variance.npz'
    sp.save_npz(str(variance_file), variance_matrix)
    print(f"\n✓ Saved variance matrix: {variance_file.name}")
    
    # Save variance lookup
    lookup_file = results_dir / f'{metapath}_variance_lookup.pkl'
    lookup_data = {
        'source_bins': source_bins,
        'target_bins': target_bins,
        'variance_lookup': variance_lookup,
        'source_degrees': source_degrees,
        'target_degrees': target_degrees
    }
    with open(lookup_file, 'wb') as f:
        pickle.dump(lookup_data, f)
    print(f"✓ Saved variance lookup: {lookup_file.name}")
    
    # Store stats
    all_variance_stats.append({
        'metapath': metapath,
        'edge1': edge1,
        'edge2': edge2,
        'n_perms': n_variance_perms,
        'matrix_shape': f"{shape[0]}x{shape[1]}",
        'n_nonzero_variance': variance_matrix.nnz,
        'min_variance': float(variance_matrix.data.min()) if variance_matrix.nnz > 0 else 0.0,
        'max_variance': float(variance_matrix.data.max()) if variance_matrix.nnz > 0 else 0.0,
        'mean_variance': float(variance_matrix.data.mean()) if variance_matrix.nnz > 0 else 0.0,
        'n_source_bins': len(source_bins),
        'n_target_bins': len(target_bins),
        'n_bin_combinations': len(variance_lookup)
    })

print(f"\n{'='*70}")
print(f"VARIANCE COMPUTATION COMPLETE")
print(f"{'='*70}")

## Summary Results

In [ ]:
# Create summary DataFrame
summary_df = pd.DataFrame(all_variance_stats)

print("\n" + "="*100)
print("VARIANCE ESTIMATION SUMMARY")
print("="*100)
print(summary_df.to_string(index=False))

# Save summary
summary_df.to_csv(results_dir / 'summary.csv', index=False)
print(f"\n✓ Saved summary: {results_dir / 'summary.csv'}")

# Overall statistics
print(f"\n{'='*100}")
print(f"OVERALL STATISTICS:")
print(f"  Total metapaths: {len(summary_df)}")
print(f"  Permutations used: {n_variance_perms} ({min(variance_perms)}-{max(variance_perms)})")
print(f"  Mean non-zero variances per metapath: {summary_df['n_nonzero_variance'].mean():.0f}")
print(f"  Mean variance (across all metapaths): {summary_df['mean_variance'].mean():.4f}")
print(f"  Mean source bins: {summary_df['n_source_bins'].mean():.1f}")
print(f"  Mean target bins: {summary_df['n_target_bins'].mean():.1f}")
print(f"{'='*100}\n")

## Visualizations

In [ ]:
# Plot: Variance statistics by metapath
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Mean variance
x = np.arange(len(summary_df))
ax1.bar(x, summary_df['mean_variance'], color='steelblue', alpha=0.7)
ax1.set_xlabel('Metapath', fontsize=12, fontweight='bold')
ax1.set_ylabel('Mean Variance', fontsize=12, fontweight='bold')
ax1.set_title('Mean Variance by Metapath', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(summary_df['metapath'], rotation=45, ha='right')
ax1.grid(axis='y', alpha=0.3)

# Number of bins
width = 0.35
ax2.bar(x - width/2, summary_df['n_source_bins'], width, label='Source bins', alpha=0.7)
ax2.bar(x + width/2, summary_df['n_target_bins'], width, label='Target bins', alpha=0.7)
ax2.set_xlabel('Metapath', fontsize=12, fontweight='bold')
ax2.set_ylabel('Number of Degree Bins', fontsize=12, fontweight='bold')
ax2.set_title('Adaptive Degree Bins by Metapath', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(summary_df['metapath'], rotation=45, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(results_dir / 'variance_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved plot: {results_dir / 'variance_summary.png'}")

In [ ]:
# Plot: Variance distribution for one example metapath
example_metapath = metapaths_2hop[0]['name']
variance_file = results_dir / f'{example_metapath}_variance.npz'
variance_matrix = sp.load_npz(str(variance_file))

fig, ax = plt.subplots(figsize=(10, 6))

# Histogram of variance values (log scale)
variance_values = variance_matrix.data[variance_matrix.data > 0]
ax.hist(np.log10(variance_values), bins=50, edgecolor='black', alpha=0.7, color='coral')
ax.set_xlabel('log₁₀(Variance)', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax.set_title(f'Variance Distribution: {example_metapath}', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add statistics
textstr = f'n = {len(variance_values):,}\n'
textstr += f'min = {variance_values.min():.2e}\n'
textstr += f'median = {np.median(variance_values):.2e}\n'
textstr += f'max = {variance_values.max():.2e}'
ax.text(0.98, 0.97, textstr, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(results_dir / 'variance_distribution_example.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved plot: {results_dir / 'variance_distribution_example.png'}")

## Test Variance Lookup

In [ ]:
# Test variance lookup function
def get_variance_for_pair(source_degree, target_degree, lookup_data):
    """
    Get variance for a (source_degree, target_degree) pair.
    
    Args:
        source_degree: Source node degree
        target_degree: Target node degree
        lookup_data: Loaded variance lookup data
    
    Returns:
        float: Variance estimate
    """
    source_bins = lookup_data['source_bins']
    target_bins = lookup_data['target_bins']
    variance_lookup = lookup_data['variance_lookup']
    
    # Find bin indices
    src_bin_idx = get_bin_index(source_degree, source_bins)
    tgt_bin_idx = get_bin_index(target_degree, target_bins)
    
    if src_bin_idx == -1 or tgt_bin_idx == -1:
        return 0.0
    
    key = (src_bin_idx, tgt_bin_idx)
    if key in variance_lookup:
        return variance_lookup[key]['variance']
    else:
        return 0.0

# Test with first metapath
example_metapath = metapaths_2hop[0]['name']
lookup_file = results_dir / f'{example_metapath}_variance_lookup.pkl'

with open(lookup_file, 'rb') as f:
    lookup_data = pickle.load(f)

# Test a few degree pairs
print(f"\nTesting variance lookup for {example_metapath}:\n")
test_pairs = [
    (10, 50),
    (100, 200),
    (5, 10),
    (500, 1000)
]

for src_deg, tgt_deg in test_pairs:
    var = get_variance_for_pair(src_deg, tgt_deg, lookup_data)
    print(f"  source_deg={src_deg:4d}, target_deg={tgt_deg:4d} → variance={var:.4f}")

print("\n✓ Variance lookup test successful")

## Conclusion

In [ ]:
print("\n" + "="*70)
print("VARIANCE ESTIMATION COMPLETE")
print("="*70)

print(f"\nComputed variance for {len(metapaths_2hop)} metapaths")
print(f"Using permutations {variance_perms_start}-{variance_perms_end}")

print(f"\nResults saved to:")
print(f"  - {results_dir / '*_variance.npz'} (variance matrices)")
print(f"  - {results_dir / '*_variance_lookup.pkl'} (degree bin lookups)")
print(f"  - {results_dir / 'summary.csv'}")
print(f"  - {results_dir / '*.png'} (plots)")

print(f"\n{'='*70}")
print(f"NEXT STEP: Run notebook 20_anomaly_detection.ipynb")
print(f"  → Use variance estimates to compute z-scores")
print(f"  → Identify metapaths with biological signal")
print(f"{'='*70}\n")